# 01 — Linear, GELU and GPT-2 small's MLP (768 → 3072 → 768)

Every GPT-2 small block contains an MLP that expands each 768-dimensional token vector to 3,072 dimensions, applies GELU, and projects back to 768. The 12 MLPs hold **56,669,184 parameters, 45.5% of the whole model**.

In this notebook:

1. derive the backward pass of a Linear layer with index notation, first for 2-D inputs, then for GPT's `(B, T, C)` inputs;
2. understand *why* the formulas look the way they do (outer products summed over tokens);
3. the activation functions: ReLU, and GELU (what GPT-2 actually uses), with their derivatives;
4. assemble GPT-2's MLP (`768 → 3072 → 768`) and gradient-check it, on a tiny copy and at full size;
5. GPT-2's initialization scheme;
6. the bugs in the `Linear` class of your script.

In [ ]:
import math
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=120)

def numerical_grad(f, x, eps=1e-6):
    grad = np.zeros_like(x, dtype=np.float64)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        grad[i] = (fp - fm) / (2 * eps)
        it.iternext()
    return grad

def rel_error(a, b):
    return np.linalg.norm(a - b) / max(1e-12, np.linalg.norm(a) + np.linalg.norm(b))

def spot_check(f, p, g, k=20, eps=1e-5, seed=123):
    """Check k random entries of the analytic gradient g (of scalar f() w.r.t. array p) against
    central differences. Entries with near-zero gradient are skipped: there the finite difference
    is pure rounding noise, so a relative error would be meaningless."""
    r = np.random.default_rng(seed)
    candidates = np.flatnonzero(np.abs(g) > 1e-3 * np.abs(g).max())
    worst = 0.0
    for i in r.choice(candidates, size=min(k, candidates.size), replace=False):
        ii = np.unravel_index(i, p.shape)
        old = p[ii]
        p[ii] = old + eps; fp = f()
        p[ii] = old - eps; fm = f()
        p[ii] = old
        num = (fp - fm) / (2 * eps)
        worst = max(worst, abs(num - g[ii]) / (abs(num) + abs(g[ii])))
    return worst

## 1. Linear layer: derivation in 2-D

Forward pass, with $N$ rows (tokens), $C_\text{in}$ input features and $C_\text{out}$ output features:

$$
Y = XW + b,\qquad X\in\mathbb{R}^{N\times C_\text{in}},\; W\in\mathbb{R}^{C_\text{in}\times C_\text{out}},\; b\in\mathbb{R}^{C_\text{out}},\; Y\in\mathbb{R}^{N\times C_\text{out}}
$$

With indices: $\;Y_{nj} = \sum_k X_{nk}W_{kj} + b_j$.

We are given $dY$ (same shape as $Y$). We want $dW$, $db$ and $dX$.

**$dW$.** Which outputs does $W_{kj}$ affect? The term $X_{nk}W_{kj}$ appears in $Y_{nj}$ for **every** row $n$, and $\partial Y_{nj}/\partial W_{kj} = X_{nk}$. Summing over all the places it's used:

$$
dW_{kj} = \sum_n dY_{nj}\, X_{nk} = \sum_n (X^\top)_{kn}\, dY_{nj}
\quad\Longrightarrow\quad
\boxed{dW = X^\top dY}\quad (C_\text{in}\times N)(N\times C_\text{out}) = (C_\text{in}\times C_\text{out})\ ✓
$$

**$db$.** $b_j$ appears in $Y_{nj}$ for every $n$ with derivative 1:

$$
\boxed{db_j = \sum_n dY_{nj}}\qquad\text{(broadcast over rows ⇒ sum over rows)}
$$

**$dX$.** $X_{nk}$ appears in $Y_{nj}$ for every $j$ (only row $n$), with $\partial Y_{nj}/\partial X_{nk} = W_{kj}$:

$$
dX_{nk} = \sum_j dY_{nj}\, W_{kj} = \sum_j dY_{nj}(W^\top)_{jk}
\quad\Longrightarrow\quad
\boxed{dX = dY\, W^\top}\quad (N\times C_\text{out})(C_\text{out}\times C_\text{in})\ ✓
$$

**The shape shortcut.** Once you know the gradient is "some product of the other two matrices", there is only *one* way to arrange $X$ and $dY$ into a $(C_\text{in}\times C_\text{out})$ matrix, and only one way to arrange $dY$ and $W$ into $(N\times C_\text{in})$. Many people derive Linear backward purely by matching shapes. It works, but do the index derivation at least once, as above.

### Intuition for $dW = X^\top dY$

Write the product as a sum over rows:

$$
dW = \sum_{n=1}^{N} x_n\, dy_n^\top \qquad (\text{a sum of } N \text{ outer products } C_\text{in}\times C_\text{out})
$$

Each token $n$ makes its own suggestion for how to change $W$: an outer product of *what came in* ($x_n$) and *how the output should have been different* ($dy_n$). The weight is shared by all tokens, so the suggestions add up. In GPT, $N = B\cdot T$, so **every position of every sequence in the batch contributes to every weight's gradient.** That's rule 2 (fan-out ⇒ sum) in action.

The intuition for $dX = dY W^\top$: $W$ maps input space to output space, and $W^\top$ maps the "blame" in output space back into input space.

## 2. Linear on `(B, T, C)` inputs

In GPT, `x` has shape `(B, T, C_in)`. The forward pass is unchanged: `x @ W` broadcasts. For the backward pass, remember that Linear treats every token independently, so the B·T tokens are just the N rows above. We **flatten** to `(B·T, C_in)`, apply the 2-D formulas, and `dX` comes out with `dout`'s leading shape automatically.

Equivalently, `dW = np.einsum('btc,btd->cd', x, dout)`: sum over both b and t.

Design choices in the class below:

- Parameters live in a dict `self.params`, and gradients in `self.grads` with the same keys and shapes. A generic optimizer can then loop over `(name, param, grad)` triples from any layer.
- Gradients are **accumulated** (`+=`). The training loop zeros them before each backward pass.
- The bias has shape `(C_out,)`. Broadcasting adds it to every `(…, C_out)` row. (Your `(1, C_out)` also works, but then `db` must keep that shape.)
- A `dtype` argument: float64 for gradient checks, float32 for the full-size model (half the memory, about twice as fast).

In GPT-2 small, the Linear layers are `c_attn` (768 → 2304), attention `c_proj` (768 → 768), `c_fc` (768 → 3072) and MLP `c_proj` (3072 → 768). For `c_fc`, `x` is `(B, T, 768)`, `W` is `(768, 3072)`, and `x2` inside `backward` is `(B·T, 768)`.

In [ ]:
class Linear:
    def __init__(self, in_f, out_f, bias=True, std=0.02, rng=None, dtype=np.float64):
        rng = rng if rng is not None else np.random.default_rng()
        self.params = {"W": rng.normal(0.0, std, (in_f, out_f)).astype(dtype)}
        if bias:
            self.params["b"] = np.zeros(out_f, dtype=dtype)
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}
        self.x = None

    def forward(self, x):                  # x: (..., in_f)
        self.x = x                         # cache for backward
        out = x @ self.params["W"]         # (..., out_f)
        if "b" in self.params:
            out = out + self.params["b"]
        return out

    def backward(self, dout):              # dout: (..., out_f)
        x2 = self.x.reshape(-1, self.x.shape[-1])     # (N, in_f),  N = product of leading dims
        d2 = dout.reshape(-1, dout.shape[-1])         # (N, out_f)
        self.grads["W"] += x2.T @ d2                  # (in_f, out_f)
        if "b" in self.params:
            self.grads["b"] += d2.sum(axis=0)         # (out_f,)
        return dout @ self.params["W"].T              # (..., in_f)

    def parameters(self, prefix=""):
        return [(prefix + k, self.params[k], self.grads[k]) for k in self.params]

### Gradient check

We use the `L = sum(out * G)` trick from notebook 00, so `dout = G`. The inputs are deliberately 3-D `(B, T, C)`.

In [ ]:
rng = np.random.default_rng(0)
B, T, C_in, C_out = 2, 3, 4, 5
lin = Linear(C_in, C_out, std=1.0, rng=rng)
lin.params["b"][:] = rng.normal(size=C_out)       # non-zero bias so it's actually tested
x = rng.normal(size=(B, T, C_in))
G = rng.normal(size=(B, T, C_out))

f = lambda: np.sum(lin.forward(x) * G)

f()
dx = lin.backward(G)
print("dx rel err:", rel_error(dx, numerical_grad(f, x)))
for name, p, g in lin.parameters():
    print(f"d{name} rel err:", rel_error(g, numerical_grad(f, p)))

### The same check at GPT-2 small's size: `c_fc`, 768 → 3072

`W` alone has 768 × 3072 = 2,359,296 entries, plus 3,072 biases, so a full numerical check would take 4.7 million forward passes. Instead we compare 20 randomly chosen entries of each gradient. Each forward pass here is cheap (B=2, T=4).

In [ ]:
c_fc = Linear(768, 3072, rng=rng)                  # GPT-2 small's c_fc, std 0.02
x = rng.normal(size=(2, 4, 768))
G = rng.normal(size=(2, 4, 3072))
f = lambda: np.sum(c_fc.forward(x) * G)
f()
dx = c_fc.backward(G)
print(f"c_fc parameters: {sum(p.size for _, p, _ in c_fc.parameters()):,}")
print("dx spot rel err:", spot_check(f, x, dx))
for name, p, g in c_fc.parameters():
    print(f"d{name} spot rel err:", spot_check(f, p, g))

## 3. Activations

### ReLU

$\text{ReLU}(z)=\max(0,z)$, with derivative $1$ if $z>0$ and $0$ otherwise. The derivative at exactly 0 is undefined; any value in $[0,1]$ is fine in practice. Backward: `dz = dout * (z > 0)`. Your script's `np.maximum(0, np.sign(x)) * prev_grad` computes exactly this, so it's correct.

### GELU — what GPT-2 uses

$$
\text{GELU}(x) = x\,\Phi(x),
$$

where $\Phi$ is the standard normal CDF. Intuition: instead of a hard gate (keep $x$ if $x>0$), the input is scaled by the *probability* that a standard normal variable is below $x$, which gives a smooth ReLU. GPT-2 uses the tanh approximation (to load its weights later, you must use the same one):

$$
\text{GELU}(x) \approx \tfrac12 x\,(1+\tanh u),\qquad u = \sqrt{2/\pi}\,\big(x + 0.044715\,x^3\big).
$$

**Derivative.** By the product and chain rules, with $t=\tanh u$ and $\frac{d}{du}\tanh u = 1-t^2$:

$$
\frac{d}{dx}\text{GELU}(x) = \tfrac12(1+t) \;+\; \tfrac12 x\,(1-t^2)\,\frac{du}{dx},
\qquad
\frac{du}{dx} = \sqrt{2/\pi}\,\big(1 + 3\cdot 0.044715\,x^2\big).
$$

GELU acts elementwise, so its Jacobian is diagonal and the backward pass is just `dx = dout * gelu'(x)`. We cache `x` and `t` in the forward pass so the backward pass doesn't recompute the tanh.

In [ ]:
class GELU:
    C = math.sqrt(2.0 / math.pi)      # a Python float, so float32 inputs stay float32

    def forward(self, x):
        self.x = x
        self.t = np.tanh(self.C * (x + 0.044715 * x * x * x))   # x*x*x is faster than x**3
        return 0.5 * x * (1.0 + self.t)

    def backward(self, dout):
        x, t = self.x, self.t
        du = self.C * (1.0 + 3 * 0.044715 * x * x)
        return dout * (0.5 * (1.0 + t) + 0.5 * x * (1.0 - t * t) * du)

    def parameters(self, prefix=""):
        return []

gelu = GELU()
x = rng.normal(size=(2, 3, 4)) * 2
G = rng.normal(size=x.shape)
f = lambda: np.sum(gelu.forward(x) * G)
f()
print("GELU dx rel err:", rel_error(gelu.backward(G), numerical_grad(f, x)))

z = np.linspace(-3, 3, 7)
print("x      :", z)
print("GELU(x):", gelu.forward(z))
print("ReLU(x):", np.maximum(0, z))

Notice that GELU is slightly negative for negative inputs (minimum ≈ −0.17 near x ≈ −0.75) and smooth everywhere. Its gradient is never exactly zero, unlike ReLU's, which is zero on the whole negative half-line (the "dead ReLU" problem).

## 4. GPT-2 small's MLP (feed-forward network)

$$
\text{MLP}(x) = \text{GELU}(x W_{fc} + b_{fc})\,W_{proj} + b_{proj},\qquad W_{fc}\in\mathbb{R}^{768\times 3072},\; W_{proj}\in\mathbb{R}^{3072\times 768}
$$

| layer | shape | weights | biases | total |
|---|---|---|---|---|
| `c_fc` | 768 → 3072 | 2,359,296 | 3,072 | 2,362,368 |
| `c_proj` | 3072 → 768 | 2,359,296 | 768 | 2,360,064 |
| **one MLP** | | | | **4,722,432** |
| **12 MLPs** | | | | **56,669,184** |

GPT-2 names the layers `c_fc` and `c_proj`, and we keep those names so pretrained weights could be loaded later. The hidden width 4C is a design choice from the original Transformer paper. It means the MLP holds about 2/3 of each block's parameters ($8C^2$ vs. $4C^2$ for attention).

Shapes for a batch: `(B, T, 768)` → `c_fc` → `(B, T, 3072)` → GELU → `(B, T, 3072)` → `c_proj` → `(B, T, 768)`.

The backward pass is the forward pass in reverse, each layer handing its returned gradient to the previous one:

```
forward : x --c_fc--> h --gelu--> a --c_proj--> out
backward: dx <--c_fc-- dh <--gelu-- da <--c_proj-- dout
```

In [ ]:
class MLP:
    def __init__(self, C, n_layer, rng, dtype=np.float64):
        self.c_fc = Linear(C, 4 * C, rng=rng, dtype=dtype)
        self.gelu = GELU()
        # GPT-2: residual-output projections get std scaled by 1/sqrt(2 * n_layer) (see section 5)
        self.c_proj = Linear(4 * C, C, std=0.02 / math.sqrt(2 * n_layer), rng=rng, dtype=dtype)

    def forward(self, x):
        return self.c_proj.forward(self.gelu.forward(self.c_fc.forward(x)))

    def backward(self, dout):
        return self.c_fc.backward(self.gelu.backward(self.c_proj.backward(dout)))

    def parameters(self, prefix=""):
        return self.c_fc.parameters(prefix + "c_fc.") + self.c_proj.parameters(prefix + "c_proj.")

In [ ]:
B, T, C = 2, 3, 4
mlp = MLP(C, n_layer=1, rng=rng)
for _, p, _ in mlp.parameters():          # bigger random values so the check isn't trivially tiny
    p[...] = rng.normal(size=p.shape)
x = rng.normal(size=(B, T, C))
G = rng.normal(size=(B, T, C))
f = lambda: np.sum(mlp.forward(x) * G)

f()
dx = mlp.backward(G)
print("dx rel err:", rel_error(dx, numerical_grad(f, x)))
for name, p, g in mlp.parameters():
    print(f"d{name:10} rel err: {rel_error(g, numerical_grad(f, p)):.2e}")

That was a tiny MLP (C = 4) with every gradient checked. Now the real one: **C = 768, hidden 3072, initialized exactly as in GPT-2 small (12 layers)**, spot-checked.

In [ ]:
mlp = MLP(768, n_layer=12, rng=rng)
print(f"MLP parameters: {sum(p.size for _, p, _ in mlp.parameters()):,}   (x 12 blocks = {12 * sum(p.size for _, p, _ in mlp.parameters()):,})")
x = rng.normal(size=(2, 4, 768))
G = rng.normal(size=(2, 4, 768))
f = lambda: np.sum(mlp.forward(x) * G)
f()
dx = mlp.backward(G)
print("dx spot rel err:", spot_check(f, x, dx))
for name, p, g in mlp.parameters():
    print(f"d{name:10} spot rel err: {spot_check(f, p, g):.2e}")

## 5. GPT-2 initialization

- All Linear weights and both embeddings: $\mathcal{N}(0,\,0.02^2)$. Biases: 0. LayerNorm: gain 1, bias 0.
- **The output projections that write into the residual stream** (`attn.c_proj` and `mlp.c_proj`) use std $0.02/\sqrt{2L}$. For GPT-2 small, $L=12$: $0.02/\sqrt{24} \approx 0.00408$.

Why the extra factor? Each block adds two terms to the residual stream: $x_{l+1} = x_l + \text{attn}(\cdot) + \text{mlp}(\cdot)$. After $L$ blocks, $2L$ roughly independent terms have been summed, and variances of independent terms add. Scaling each term's std by $1/\sqrt{2L}$ keeps the total variance of the residual stream about the same no matter how deep the model is. (Your `res_weight_init_scale` argument was heading in this direction.)

Why such small values at all? With small weights the initial logits are near zero, so the model starts by predicting a nearly **uniform** distribution over the 50,257 tokens. The initial loss is then close to $\ln 50257 \approx 10.82$, a clean, known starting point (notebook 02 computes the exact expected value, ≈ 10.98).

## 6. Bugs in your script's `Linear`

```python
self.grad_weights = np.zeros[self.weights.shape]   # (1) square brackets: TypeError
...
def forward(self, inputs):
    self.units = inputs                             # (2) stored as .units, but __init__ defines .inputs
    if self.include_bias:
        return inputs @ self.weights * self.bias    # (3) * instead of +, and bias is zeros -> output always 0
```

1. `np.zeros[...]` must be `np.zeros(...)`.
2. Whatever name you cache the input under, `backward` must read the same name. Caching the input is the whole point: `dW = x.T @ dout` needs it.
3. `* self.bias` multiplies by a zero bias, so the layer outputs all zeros and every gradient vanishes. It must be `+`.

Also, `np.random.randn(...) * 0.02` is fine (std 0.02), and it matches GPT-2.

## Summary

| Layer | Forward | Backward |
|---|---|---|
| Linear | `y = x @ W + b` | `dW = x2.T @ d2` (flattened over B,T), `db = d2.sum(0)`, `dx = dout @ W.T` |
| MLP (GPT-2 small) | `768 → 3072 → 768` | 4,722,432 parameters per block, 56,669,184 in total |
| ReLU | `max(0, x)` | `dout * (x > 0)` |
| GELU | `0.5 x (1 + tanh u)` | `dout * [0.5(1+t) + 0.5 x (1-t²) u']` |
| MLP | `c_proj(gelu(c_fc(x)))` | chain the three backwards in reverse |

Next: **02 — Softmax and cross-entropy**, the place where the very first gradient of the backward pass is created.